# Lab R3 — Retrieval & Reranking: fixing the compliance metric

**Curriculum §4 · Lab R3**

Hold chunking and the model fixed. Vary the **retriever** — dense, sparse (BM25), hybrid — then add a **cross-encoder reranker**, and prove that a **version metadata filter** is what actually rescues `version_correct` (the metric that decides whether you ever cite the superseded `pol-v2`).

## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and clones the shared `common/` package from GitHub.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade + a restart.

In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())          # fix Colab PIL._typing._Ink clash
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu rank_bm25 langchain langchain-community langchain-groq langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF pandas'.split())

    # The repo is public — clone it to get the shared common/ package.
    REPO = pathlib.Path('/content/labpractice')
    if not (REPO/'common'/'harness.py').exists():
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/AICareerstack26/labpractice', str(REPO)])
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets (🔑 sidebar) -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e:
        print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))          # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))              # make `common` importable locally
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| in force:', [d['id'] for d in current_docs()], '| superseded:', SUPERSEDED_IDS)

## 1 · Build the chunk index — naive (all docs) vs version-filtered

The trap: if you index **every** document, the retriever can surface `pol-v2` (superseded) and your bank quotes a policy that is no longer in force. `current_docs()` applies the metadata filter (in-force, newest, UK).

In [ ]:
import numpy as np, re
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)

def chunks_from(docs, contextual=True):
    out = []
    for d in docs:
        for i, piece in enumerate(splitter.split_text(d['text'])):
            body = f"[{d['doc']} {d['version']}] {piece}" if contextual else piece
            out.append(dict(id=d['id'], doc=d['doc'], version=d['version'],
                            effective=d['effective'], chunk_i=i, text=body))
    return out

NAIVE   = chunks_from(DOCS)             # includes pol-v2  -> version trap
FILTERED = chunks_from(current_docs())  # metadata-filtered -> compliant
print('naive chunks   :', len(NAIVE), ' ids:', sorted({c['id'] for c in NAIVE}))
print('filtered chunks:', len(FILTERED), ' ids:', sorted({c['id'] for c in FILTERED}))

## 2 · Three retrievers + a reranker

- **dense** — cosine over `bge` embeddings
- **bm25** — lexical, nails exact terms (rates, IDs)
- **hybrid** — Reciprocal Rank Fusion of the two
- **+rerank** — a cross-encoder reorders the top candidates

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq

llm   = ChatGroq(model=GEN_MODEL,   temperature=0)
judge = ChatGroq(model=JUDGE_MODEL, temperature=0)
_emb, _rr = {}, {}
def emb(n):
    if n not in _emb: _emb[n] = SentenceTransformer(n)
    return _emb[n]
def reranker(n):
    if n not in _rr: _rr[n] = CrossEncoder(n)
    return _rr[n]

def dense_rank(chunks, M, qv):            # returns indices best->worst
    return list(np.argsort(-(M @ qv)))
def bm25_rank(bm, query):
    return list(np.argsort(-bm.get_scores(query.split())))
def rrf(*rankings, k=60):
    score = {}
    for r in rankings:
        for pos, idx in enumerate(r): score[idx] = score.get(idx, 0) + 1.0/(k+pos+1)
    return [i for i,_ in sorted(score.items(), key=lambda x:-x[1])]

PROMPT = ("You are Meridian Bank's credit policy copilot. Answer ONLY from context. "
          "Cite the id in square brackets. If absent, reply INSUFFICIENT_CONTEXT."
          "\n\nContext:\n{ctx}\n\nQuestion: {q}")

## 3 · One pipeline, switched by `cfg['retriever']`

In [ ]:
@observe(name='rag.request')
def pipeline(query, cfg):
    chunks = FILTERED if cfg.get('version_filter', True) else NAIVE
    m = emb(cfg['embed_model'])
    M = m.encode([c['text'] for c in chunks], normalize_embeddings=True)
    qv = m.encode([query], normalize_embeddings=True)[0]
    bm = BM25Okapi([c['text'].split() for c in chunks])

    if cfg['retriever'] == 'dense':   order = dense_rank(chunks, M, qv)
    elif cfg['retriever'] == 'bm25':  order = bm25_rank(bm, query)
    else:                             order = rrf(dense_rank(chunks, M, qv), bm25_rank(bm, query))

    cand = order[:max(cfg['k'], cfg.get('rerank_pool', cfg['k']))]
    if cfg.get('reranker'):
        rr = reranker(RERANKERS[cfg['reranker']])
        s  = rr.predict([(query, chunks[i]['text']) for i in cand])
        cand = [cand[i] for i in np.argsort(-s)]
    top = cand[:cfg['k']]

    hits = [dict(chunks[i]) for i in top]
    ctx  = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    ans  = llm.invoke(PROMPT.format(ctx=ctx, q=query)).content
    span_meta(retriever=cfg['retriever'], reranker=cfg.get('reranker'),
              version_filter=cfg.get('version_filter'), returned=[h['id'] for h in hits])
    trace_meta(tags=[cfg['hash']], config=cfg)
    return dict(answer=ans, hits=hits)

## 4 · The sweep — and the one row that matters

In [ ]:
rows = []
grid = [
  dict(retriever='dense', version_filter=False),                       # the naive trap
  dict(retriever='dense', version_filter=True),
  dict(retriever='bm25',  version_filter=True),
  dict(retriever='hybrid',version_filter=True),
  dict(retriever='hybrid',version_filter=True, reranker='bge', rerank_pool=6),
]
for g in grid:
    cfg = make_config(embed_model=EMBED_MODELS['base'], k=3, **g)
    print('running', g)
    rows.append(evaluate(cfg, pipeline, judge=judge))

leaderboard(rows, sort_by='version_correct')[
  ['config','retriever','version_filter','reranker','hit_at_k','mrr','version_correct','citation','latency_s']]

## 5 · What you should conclude

- **`version_filter=False` tanks `version_correct`** — the dense retriever happily returns `pol-v2`. No amount of reranking fixes a compliance failure you let into the candidate set. **Filter at retrieval, not after.**
- **BM25 alone** wins the `table` question (exact `75% LTV` match) but loses paraphrase; **hybrid** gets both.
- **The reranker** lifts `mrr` — the right chunk moves to position 1, which improves grounding and citations.

> **Architectural takeaway:** in a regulated domain, retrieval is a *filtering* problem before it is a ranking problem. The metadata filter is the single highest-leverage line in this notebook.

**Next →** `lab04` (inference optimization)